# PocketLM: one GPU experiment at a time

The rank-8 LoRA model, its held-out evaluation, preference-pair sample, and GGUF comparisons already ran locally. The NF4 QLoRA run also completed on Colab and its result ZIP was imported and verified on 16 October 2026; do not rerun QLoRA just to continue this project. The upload bundle includes the completed rank-8 adapter. This notebook runs **at most one** remaining experiment per session, then downloads its adapter and metrics immediately. Select an experiment in the `RUN_KIND` code cell below; the default `none` checks only the upload and spends no GPU training time. You can use a CPU runtime for that check. The rank-16 and rank-32 LoRA runs can also be done on a sufficiently provisioned local MPS machine. Only NF4 QLoRA requires CUDA in this implementation. General evaluation, DPO, quantization, and model comparison can be done locally after the selected adapter comes back. Do not claim a full comparison from a shortened diagnostic run.

On the laptop, rebuild `colab/pocketlm_colab_bundle.zip` with `.venv/bin/python -m colab.build_bundle` and upload it below. The bundle contains processed public/synthetic rows and the completed rank-8 adapter; keep it private. No model hub upload or paid API is used.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
assert len(uploaded) == 1 and next(iter(uploaded)).endswith('.zip')
with zipfile.ZipFile(next(iter(uploaded))) as archive:
    assert all(not Path(name).is_absolute() and '..' not in Path(name).parts for name in archive.namelist())
    archive.extractall('/content')
%cd /content/pocketlm
assert Path('runs/sft_r8_lr1e4_2k/adapter/adapter_model.safetensors').is_file()
print('Local rank-8 baseline is ready; no retraining is needed.')

In [ ]:
print('Bundle verified. Choose a single experiment below to install dependencies and train.')

## Choose one experiment

For the next missing comparison, set `RUN_KIND` to `r16` or `r32`, one complete LoRA run per Colab session. `qlora` remains available only to deliberately reproduce the already completed CUDA-only NF4 experiment. `none` checks the upload and stops before installing packages or touching a GPU. For a real run, select a T4 GPU runtime first, change `RUN_KIND`, and run from the upload cell through the download cell in order. Each complete run uses the same deterministic first 2,000 training rows and two configured epochs. There is no automatic chain of training jobs. QLoRA keeps only its latest complete checkpoint every 100 optimizer steps; rank-16/32 still save no intermediate checkpoint. To resume QLoRA in a fresh runtime, set `RESUME_CHECKPOINT=True` and upload the checkpoint ZIP when prompted. In the same runtime, rerunning the QLoRA cell resumes from its latest complete checkpoint automatically. The checkpoint exists only in ephemeral Colab storage until you download it; a forced runtime disconnect can lose it. Download a completed result ZIP before closing the runtime.

In [ ]:
from pathlib import Path
import subprocess
RUN_KIND = 'none'  # Next missing run: 'r16' or 'r32'; 'qlora' repeats completed work.
RESUME_CHECKPOINT = False  # True only when uploading your saved QLoRA checkpoint ZIP.
jobs = {
    'qlora': ('configs/sft_qlora.yaml', 'runs/sft_qlora_r16_2k', []),
    'r16': ('configs/sft_lora.yaml', 'runs/sft_r16_lr2e4_2k', ['--rank', '16', '--learning-rate', '0.0002']),
    'r32': ('configs/sft_lora.yaml', 'runs/sft_r32_lr2e4_2k', ['--rank', '32', '--learning-rate', '0.0002']),
}
if RUN_KIND == 'none':
    print('Setup complete. Change RUN_KIND to one experiment when ready.')
else:
    if RUN_KIND not in jobs:
        raise ValueError(f'Unknown RUN_KIND: {RUN_KIND}')
    config, output_name, extra = jobs[RUN_KIND]
    output = Path(output_name)
    complete = (output / 'metrics.json').is_file() and (output / 'adapter/adapter_model.safetensors').is_file()
    if complete:
        print(f'Already complete: {output}')
    else:
        checkpoint = None
        if RUN_KIND == 'qlora':
            from colab.checkpoint_bundle import latest_complete, restore
            if RESUME_CHECKPOINT:
                from google.colab import files
                uploaded_checkpoint = files.upload()
                assert len(uploaded_checkpoint) == 1 and next(iter(uploaded_checkpoint)).endswith('.zip')
                checkpoint = restore(Path(next(iter(uploaded_checkpoint))), output)
            elif output.exists() and any(output.iterdir()):
                checkpoint = latest_complete(output)
            if checkpoint:
                print(f'Resuming QLoRA from {checkpoint}; earlier unsaved steps must run again.')
        elif output.exists() and any(output.iterdir()):
            raise RuntimeError(f'Incomplete {output}; this LoRA run has no restart checkpoint. Use a fresh runtime or remove that partial directory before intentionally starting again.')
        subprocess.run(['nvidia-smi'], check=True)
        subprocess.run(['pip', 'install', '-q', 'uv'], check=True)
        subprocess.run(['uv', 'sync', '--locked', '--python', '3.12', '--extra', 'cuda'], check=True)
        subprocess.run(['.venv/bin/python', '-c', 'import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))'], check=True)
        command = ['.venv/bin/python', '-m', 'train.sft', '--config', config,
                   '--output', output_name, '--max-rows', '2000', *extra]
        if checkpoint:
            command += ['--resume-from-checkpoint', str(checkpoint)]
        subprocess.run(command, check=True)
        assert (output / 'metrics.json').is_file() and (output / 'adapter/adapter_model.safetensors').is_file()
    print((output / 'metrics.json').read_text())

## If you interrupt QLoRA before it finishes

Set `EXPORT_CHECKPOINT=True` in the next cell and run it while this Colab runtime is still alive. It downloads `pocketlm_qlora_checkpoint.zip`, containing one adapter checkpoint with optimizer, scheduler, RNG, and trainer state (at most 512 MiB uncompressed). The most recent 0–99 unsaved steps will be repeated on resume. A checkpoint cannot be recovered after an expired runtime has deleted `/content`; exporting is a manual step before leaving. Keep the downloaded ZIP private.

In [ ]:
EXPORT_CHECKPOINT = False  # Set True after interrupting a QLoRA run.
if EXPORT_CHECKPOINT:
    from pathlib import Path
    from google.colab import files
    from colab.checkpoint_bundle import export_latest
    checkpoint_zip = export_latest(Path('runs/sft_qlora_r16_2k'),
                                   Path('/content/pocketlm_qlora_checkpoint.zip'))
    files.download(str(checkpoint_zip))

In [ ]:
import shutil
from google.colab import files
if RUN_KIND == 'none':
    print('No experiment selected; nothing to download.')
else:
    _, output_name, _ = jobs[RUN_KIND]
    output = Path(output_name)
    assert (output / 'metrics.json').is_file() and (output / 'adapter/adapter_model.safetensors').is_file(), 'Training is incomplete; export a QLoRA checkpoint instead.'
    archive = shutil.make_archive(f'/content/pocketlm_gpu_results_{RUN_KIND}', 'zip',
                                  root_dir='/content/pocketlm', base_dir=output_name)
    print(archive)
    files.download(archive)

## Back on the laptop

The browser downloads exactly one `pocketlm_gpu_results_qlora.zip`, `pocketlm_gpu_results_r16.zip`, or `pocketlm_gpu_results_r32.zip`, matching `RUN_KIND`. Place the ZIP in this project's `colab/` folder and extract its `runs/...` folder into this project. Confirm `runs/<selected_run>/metrics.json` and `runs/<selected_run>/adapter/adapter_model.safetensors` exist before comparing models. Keep the rank-8 baseline and all completed runs. The local machine can select the best adapter by **validation loss only** with `python -m train.select_best` on the completed run directories, then perform held-out evaluation. Preference pairs must be sampled again from whichever adapter is selected before a DPO run; the rank-8 pairs already sampled locally are only nine examples and do not justify a broad DPO quality claim. Local CPU GGUF evaluation and latency runs do not require Colab. See `README.md` and `RESULTS.md` for reproducible commands and measured results.